In [31]:
from pathlib import Path

from langchain_huggingface import HuggingFaceEmbeddings

from llama_index.core import Document
from llama_index.core.node_parser import SemanticSplitterNodeParser
from llama_index.embeddings.langchain import LangchainEmbedding

In [32]:
langchain_embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en-v1.5"
)

embedding_model = LangchainEmbedding(langchain_embedding_model)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [33]:
file_path = Path("NVDA.txt")
text = file_path.read_text(encoding="utf-8")

doc = Document(
    text=text,
    metadata={
        "source": str(file_path),
        "topic": "stocks_news"
    }
)

In [34]:
splitter = SemanticSplitterNodeParser(
    embed_model=embedding_model,
    buffer_size=1,
    breakpoint_percentile_threshold=95,
)

In [35]:
nodes = splitter.get_nodes_from_documents([doc])

print(f"Number of chunks: {len(nodes)}")

Number of chunks: 3


In [39]:
chunk_texts = [node.get_content() for node in nodes]

In [41]:
chunk_embeddings = langchain_embedding_model.embed_documents(chunk_texts)

In [42]:
query = "Why might Nvidia be overvalued?"
query_embedding = langchain_embedding_model.embed_query(query)

In [43]:
import numpy as np

def cosine_similarity(a, b):
    a = np.array(a)
    b = np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

In [44]:
def retrieve_top_k(query: str, chunk_texts: list[str], chunk_embeddings: list[list[float]], embed_model, k: int = 2):
    query_embedding = embed_model.embed_query(query)

    scores = [
        cosine_similarity(query_embedding, chunk_embedding)
        for chunk_embedding in chunk_embeddings
    ]

    ranked_results = sorted(
        list(enumerate(scores)),
        key=lambda x: x[1],
        reverse=True
    )

    top_results = ranked_results[:k]

    retrieved_chunks = []
    for idx, score in top_results:
        retrieved_chunks.append({
            "chunk_id": idx + 1,
            "score": score,
            "text": chunk_texts[idx]
        })

    return retrieved_chunks

In [45]:
def build_rag_prompt(query: str, retrieved_chunks: list[dict]) -> str:
    context_blocks = []

    for item in retrieved_chunks:
        context_blocks.append(
            f"[Chunk {item['chunk_id']} | similarity={item['score']:.4f}]\n{item['text']}"
        )

    context = "\n\n".join(context_blocks)

    prompt = f"""
You are a grounded financial news assistant.

Answer the user's question using ONLY the provided context.
Do not make up facts not present in the context.
If the context is insufficient, say so clearly.
Do not give financial advice or recommend buying/selling.
Provide a concise, neutral summary.

User question:
{query}

Retrieved context:
{context}

Answer:
""".strip()

    return prompt

In [46]:
def ask_rag(query: str, chunk_texts: list[str], chunk_embeddings: list[list[float]], embed_model, llm, k: int = 2):
    retrieved_chunks = retrieve_top_k(
        query=query,
        chunk_texts=chunk_texts,
        chunk_embeddings=chunk_embeddings,
        embed_model=embed_model,
        k=k
    )

    prompt = build_rag_prompt(query, retrieved_chunks)
    response = llm.invoke(prompt)

    return {
        "query": query,
        "retrieved_chunks": retrieved_chunks,
        "prompt": prompt,
        "answer": response.content
    }

In [47]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="qwen3:8b",
    temperature=0.2
)

In [48]:
query = "Why might Nvidia be overvalued?"

result = ask_rag(
    query=query,
    chunk_texts=chunk_texts,
    chunk_embeddings=chunk_embeddings,
    embed_model=langchain_embedding_model,
    llm=llm,
    k=2
)

In [49]:
for item in result["retrieved_chunks"]:
    print(f"\n=== Chunk {item['chunk_id']} | Score: {item['score']:.4f} ===\n")
    print(item["text"][:1000])


=== Chunk 1 | Score: 0.7677 ===

If you are wondering whether NVIDIA's current share price still makes sense, you are not alone. This article looks at what you are really paying for when you buy the stock.
NVIDIA shares last closed at US$177.82, with returns of 0.4% over 7 days, 3.5% over 30 days, a 5.8% decline year to date, and a 57.8% gain over 1 year, while the 3 year return is very large and over 7x over 5 years.
Recent headlines have focused heavily on NVIDIA's role in graphics chips and data center hardware for high performance computing, alongside increased attention on its position in AI related workloads. This context has kept NVIDIA at the center of market conversations about semiconductor demand, capital investment and the long term importance of its technology stack.
Our current valuation framework gives NVIDIA a valuation score of 3 out of 6. Next we will walk through what that means across different methods of valuing the company, before finishing with an even more comp

In [50]:
print(result["prompt"])

You are a grounded financial news assistant.

Answer the user's question using ONLY the provided context.
Do not make up facts not present in the context.
If the context is insufficient, say so clearly.
Do not give financial advice or recommend buying/selling.
Provide a concise, neutral summary.

User question:
Why might Nvidia be overvalued?

Retrieved context:
[Chunk 1 | similarity=0.7677]
If you are wondering whether NVIDIA's current share price still makes sense, you are not alone. This article looks at what you are really paying for when you buy the stock.
NVIDIA shares last closed at US$177.82, with returns of 0.4% over 7 days, 3.5% over 30 days, a 5.8% decline year to date, and a 57.8% gain over 1 year, while the 3 year return is very large and over 7x over 5 years.
Recent headlines have focused heavily on NVIDIA's role in graphics chips and data center hardware for high performance computing, alongside increased attention on its position in AI related workloads. This context ha

In [ ]:
print(result["answer"])